# 风机效率实时计算（台儿庄）

模型 ID `fan-efficiency-taierzhuang`｜目标 `cpu-iot`｜生命周期 `specified`。

本 Notebook 回答四件事，并给出可自动复核的断言：

1. 两个口径（Excel / 修正）到底差什么、差多少；
2. 现场数据可用到什么程度（质量标记与中断段）；
3. 双口径偏差随进气温度如何变号；
4. 四条验收标准怎么当场断言。

> 运行前提：仓库根已有 `data/fan-efficiency-taierzhuang/` 的取数产物（由 `scripts/取数探针/iot_bulk_fetch.py` 生成）。

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

WS = Path.cwd().resolve()
while not (WS / 'src' / 'model.py').exists():
    WS = WS.parent
sys.path.insert(0, str(WS / 'src'))
import model as fan  # noqa: E402

REPO = WS.parents[2]
DATA = REPO / 'data' / 'fan-efficiency-taierzhuang'

print('python', sys.version.split()[0], '| pandas', pd.__version__, '| numpy', np.__version__)
print('K =', fan.K, '| P_std =', fan.P_STD_KPA, 'kPa | T_std =', fan.T_STD_K, 'K')
print('C18 =', fan.C18, '  (C18*K/1e3 =', round(fan.C18 * fan.K / 1e3, 4), 'kPa)')
print('数据目录', DATA, '存在' if DATA.exists() else '缺失')

python 3.11.15 | pandas 2.3.3 | numpy 2.3.3
K = 0.2857 | P_std = 101.325 kPa | T_std = 293.15 K
C18 = 354635.537746665   (C18*K/1e3 = 101.3194 kPa)
数据目录 E:\beikong-code\learn-model\data\fan-efficiency-taierzhuang 存在


## 1. 契约

| 方向 | 字段 | 单位 | 说明 |
|---|---|---|---|
| 输入 | `inlet_flow` | m³/min | 风机面板显示风量（工况体积流量） |
| 输入 | `outlet_pressure` | kPa | 出口表压（即背压 Δp） |
| 输入 | `motor_power` | kW | 电机输入功率 |
| 输入 | `inlet_temperature` | ℃ | 进气温度 |
| 输入 | `elevation` | m | 海拔（唯一可调参数，默认 38.5） |
| 输出 | `eta_excel` | 1 | Excel 口径效率（**本期平台生效值**） |
| 输出 | `eta_corrected` | 1 | 修正口径效率 |
| 输出 | `wp_excel` / `wp_corrected` | kW | 两个口径的气动功率 |
| 输出 | `quality` | 枚举 | `VALID` / `STOPPED` / `ROW_INVALID` / `NO_SAMPLE` / `RATIO_INVALID` / `OUT_OF_RANGE` / `DATA_OUTAGE` |

In [2]:
ledger = pd.DataFrame(
    [
        ('一污4#', 46.1, 46.2, 42.2, 25.5),
        ('一污5#', 48.6, 46.1, 41.9, 25.8),
        ('二污2#', 48.2, 62.1, 61.3, 26.0),
    ],
    columns=['machine', 'inlet_flow', 'outlet_pressure', 'motor_power', 'inlet_temperature'],
)
ledger['elevation'] = fan.DEFAULT_ELEVATION_M
out = fan.predict(ledger)
result = pd.concat([ledger[['machine']], out.set_index(ledger.index)], axis=1)
result.round(4)

,machine,eta_excel,eta_corrected,wp_excel,wp_corrected,quality
0,一污4#,0.7177,0.7312,30.2874,30.8574,VALID
1,一污5#,0.7598,0.7749,31.8372,32.4689,VALID
2,二污2#,0.6660,0.6797,40.8277,41.6657,VALID


In [3]:
# 验收断言 ②：台账断面（自控信号台账 2026-10-16 的实时读数）
expect = {'一污4#': 0.7312, '一污5#': 0.7749, '二污2#': 0.6797}
for machine, target in expect.items():
    got = float(result.loc[result['machine'] == machine, 'eta_corrected'].iloc[0])
    assert abs(got - target) < 5e-4, (machine, got, target)
print('✅ 台账断面三条全部在 0.05 个百分点内')
print('全部质量标记：', out['quality'].unique())

✅ 台账断面三条全部在 0.05 个百分点内
全部质量标记： ['VALID']


## 2. Excel 口径与修正口径的差别

同一个压比项 `Π^K − 1`，两个口径只差**前导项的状态配对**：

- Excel：`P标准 × Q标准`（101.325 kPa × 标况流量）——等于把进口状态当成 20 ℃、1 atm；
- 修正：`P入口 × Q入口`（当地压力 × 工况流量）。

两者的桥是质量守恒 `P入口·Q入口 = P标准·Q标准·(T入口/T标准)`，所以**偏差只由进气温度决定**，与风量无关。

In [4]:
excel_case = pd.DataFrame([
    dict(motor_power=72.4, outlet_pressure=47.2, inlet_flow=93.0,
         inlet_temperature=20.0, elevation=38.5)
])
base = fan.predict(excel_case).iloc[0]

# 复现 Excel 原表的 24.8 ℃ 读数（Excel 用的是 24.8 ℃ 的工况流量，却按标准状态算）
excel_248 = fan.compute_row(93.0, 47.2, 72.4, 24.8, 38.5)

print('20 ℃ 时理论偏差 =', round(float((base.eta_excel / base.eta_corrected - 1) * 100), 4), '%')
print('24.8 ℃ 时（Excel 原表工况）')
print('  Excel 口径  Wp = %.4f kW, eta = %.4f%%' % (excel_248['wp_excel'], excel_248['eta_excel'] * 100))
print('  修正口径   Wp = %.4f kW, eta = %.4f%%' % (excel_248['wp_corrected'], excel_248['eta_corrected'] * 100))
print('偏差（Excel − 修正）= %.4f%%' % ((excel_248['eta_excel'] / excel_248['eta_corrected'] - 1) * 100))

20 ℃ 时理论偏差 = -0.0056 %
24.8 ℃ 时（Excel 原表工况）
  Excel 口径  Wp = 62.4061 kW, eta = 86.1963%
  修正口径   Wp = 63.4314 kW, eta = 87.6125%
偏差（Excel − 修正）= -1.6165%


In [5]:
# 验收断言 ①：公式复现
assert abs(excel_248['eta_excel'] - 0.861963) < 1e-5
assert abs(excel_248['eta_corrected'] - 0.876125) < 1e-5
assert abs(excel_248['wp_excel'] - 62.406094) < 1e-4
assert abs(excel_248['wp_corrected'] - 63.431445) < 1e-4
print('✅ 公式复现：86.20% / 87.61%，62.406 / 63.431 kW')

✅ 公式复现：86.20% / 87.61%，62.406 / 63.431 kW


## 3. 数据可用性

流水线顺序固定为：**先做行级有效性判定 → 时间加权聚合 → 分钟级质量标记 → 中断升级**。
运行时代码（`src/model.py`）与离线统计用的是同一份函数，避免两边口径漂移。

In [6]:
minute_path = DATA / 'derived' / 'minute-v2.parquet'
if minute_path.exists():
    minute = pd.read_parquet(minute_path)
    counts = (minute.groupby(['machine', 'quality']).size()
              .unstack(fill_value=0))
    share = counts.div(counts.sum(axis=1), axis=0)
    print('分钟级样本数与质量分布（行=机组）')
    display(pd.concat([counts, (share * 100).round(2).add_suffix('_pct')], axis=1))
else:
    print('缺少 minute-v2.parquet，请先运行 work/analysis/analyze_v2.py')
    counts = None

分钟级样本数与质量分布（行=机组）


quality,DATA_OUTAGE,NO_SAMPLE,OUT_OF_RANGE,STOPPED,VALID,DATA_OUTAGE_pct,NO_SAMPLE_pct,OUT_OF_RANGE_pct,STOPPED_pct,VALID_pct
machine,,,,,,,,,,
一污4#,1467,1355,0,27212,366471,0.37,0.34,0.0,6.86,92.43
一污5#,1467,1390,0,2784,390864,0.37,0.35,0.0,0.70,98.58
一污6#,37474,1323,0,336191,21517,9.45,0.33,0.0,84.79,5.43
二污1#,21195,1897,0,696974,48894,2.76,0.25,0.0,90.64,6.36
二污2#,21195,1895,1,485682,260187,2.76,0.25,0.0,63.16,33.84
二污3#,21195,1896,0,474545,271324,2.76,0.25,0.0,61.71,35.28


In [7]:
if minute_path.exists():
    intervals = []
    for machine, group in minute.sort_values(['machine', 'minute']).groupby('machine'):
        bad = group['quality'].isin([fan.NO_SAMPLE, fan.ROW_INVALID, fan.DATA_OUTAGE]).to_numpy()
        t = group['minute'].to_numpy()
        start = None
        for i, flag in enumerate(np.append(bad, False)):
            if flag and start is None:
                start = i
            elif not flag and start is not None:
                hours = (i - start) * 60 / 3600
                if hours > 6:
                    intervals.append((machine, pd.Timestamp(t[start]), pd.Timestamp(t[i - 1]), round(hours, 1)))
                start = None
    outages = pd.DataFrame(intervals, columns=['machine', 'start', 'end', 'hours'])
    print(outages.to_string(index=False) if len(outages) else '无中断段')

machine                     start                       end  hours
   一污4# 2026-02-07 14:53:00+00:00 2026-02-08 15:19:00+00:00   24.4
   一污5# 2026-02-07 14:53:00+00:00 2026-02-08 15:19:00+00:00   24.4
   一污6# 2026-02-07 14:53:00+00:00 2026-02-08 15:19:00+00:00   24.4
   一污6# 2026-09-17 03:17:00+00:00 2026-10-12 03:23:00+00:00  600.1
   二污1# 2025-04-30 16:00:00+00:00 2025-05-14 08:47:00+00:00  328.8
   二污1# 2026-02-07 14:53:00+00:00 2026-02-08 15:19:00+00:00   24.4
   二污2# 2025-04-30 16:00:00+00:00 2025-05-14 08:47:00+00:00  328.8
   二污2# 2026-02-07 14:53:00+00:00 2026-02-08 15:19:00+00:00   24.4
   二污3# 2025-04-30 16:00:00+00:00 2025-05-14 08:47:00+00:00  328.8
   二污3# 2026-02-07 14:53:00+00:00 2026-02-08 15:19:00+00:00   24.4


## 4. 偏差随温度变号

偏差 = `eta_excel / eta_corrected − 1`，等价于 `T标准/T入口 − 1`。
20 ℃ 附近为 0，冬天为正（Excel 高估），夏天为负（Excel 低估）。

In [8]:
if minute_path.exists():
    valid = minute[minute['quality'] == fan.VALID].copy()
    valid['dev'] = (valid['eta_excel'] / valid['eta_corrected'] - 1) * 100
    edges = [-30, 10, 15, 20, 25, 30, 35, 60]
    valid['band'] = pd.cut(valid['inlet_temperature'], edges)
    table = valid.groupby('band', observed=True).agg(
        样本=('dev', 'size'),
        效率修正中位=('eta_corrected', 'median'),
        效率Excel中位=('eta_excel', 'median'),
        偏差中位=('dev', 'median'),
    )
    table['效率修正中位'] = (table['效率修正中位'] * 100).round(2)
    table['效率Excel中位'] = (table['效率Excel中位'] * 100).round(2)
    table['偏差中位'] = table['偏差中位'].round(2)
    display(table)
else:
    print('跳过：缺少 minute-v2.parquet')

,样本,效率修正中位,效率Excel中位,偏差中位
band,,,,
"(-30, 10]",51719,70.42,73.82,4.41
"(10, 15]",122786,70.50,72.16,2.44
"(15, 20]",134348,70.14,70.83,1.00
"(20, 25]",157964,72.27,71.62,-0.93
"(25, 30]",253926,71.67,69.65,-2.69
"(30, 35]",375437,71.12,68.05,-4.17
"(35, 60]",263077,67.65,64.02,-5.44


In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

plots_dir = WS / 'plots'
plots_dir.mkdir(exist_ok=True)

temps = np.linspace(0, 45, 91)
dev = (fan.T_STD_K / (temps + 273.15) - 1) * 100
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.axhline(0, color='#999', lw=0.8)
ax.axvline(20, color='#c0392b', lw=0.8, ls='--')
ax.plot(temps, dev, color='#1f4e79', lw=2)
ax.fill_between(temps, dev, 0, where=dev > 0, color='#c0392b', alpha=0.12)
ax.fill_between(temps, dev, 0, where=dev < 0, color='#2e7d32', alpha=0.12)
ax.set_xlabel('进气温度 (℃)')
ax.set_ylabel('Excel 偏差 (%)')
ax.set_title('双口径偏差只由进气温度决定（正=Excel 高估）')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(plots_dir / 'eta-deviation-vs-temperature.png', dpi=150)
plt.close(fig)
print('图已保存 ->', plots_dir / 'eta-deviation-vs-temperature.png')
print('5 ℃: %+.2f%% | 20 ℃: %+.2f%% | 30 ℃: %+.2f%% | 40 ℃: %+.2f%%'
      % tuple((fan.T_STD_K / (t + 273.15) - 1) * 100 for t in (5, 20, 30, 40)))

图已保存 -> E:\beikong-code\learn-model\models\_drafts\fan-efficiency-taierzhuang\plots\eta-deviation-vs-temperature.png
5 ℃: +5.39% | 20 ℃: +0.00% | 30 ℃: -3.30% | 40 ℃: -6.39%


## 5. 验收断言

四条标准全部可自动复核：公式复现、台账断面、差值指纹、缺输入置空。

In [10]:
# 验收断言 ③：差值指纹
def dev_at(t):
    row = fan.compute_row(93.0, 47.2, 72.4, t, 38.5)
    return (row['eta_excel'] / row['eta_corrected'] - 1) * 100

assert abs(dev_at(20.0)) < 0.01
assert abs(dev_at(5.0) - 5.39) < 0.05
assert abs(dev_at(30.0) + 3.35) < 0.05
print('✅ 差值指纹：20 ℃ = %.4f%%，5 ℃ = %+.2f%%，30 ℃ = %+.2f%%'
      % (dev_at(20.0), dev_at(5.0), dev_at(30.0)))

# 验收断言 ④：缺输入置空
missing = pd.DataFrame([dict(inlet_flow=np.nan, outlet_pressure=47.2, motor_power=72.4,
                             inlet_temperature=24.8, elevation=38.5)])
row = fan.predict(missing).iloc[0]
assert row['quality'] == fan.NO_SAMPLE
assert np.isnan(row['eta_excel']) and np.isnan(row['eta_corrected'])
print('✅ 缺输入 ->', row['quality'], '，两个口径均置空')

✅ 差值指纹：20 ℃ = -0.0056%，5 ℃ = +5.39%，30 ℃ = -3.30%
✅ 缺输入 -> NO_SAMPLE ，两个口径均置空


In [11]:
# 验收断言 ⑤（补）：偏差与风量无关
ratios = []
for flow in (40.0, 60.0, 80.0, 100.0):
    r = fan.compute_row(flow, 47.2, 72.4 * flow / 93.0, 24.8, 38.5)
    ratios.append(r['eta_excel'] / r['eta_corrected'])
assert max(ratios) - min(ratios) < 1e-9
print('✅ 改风量 40→100 偏差恒为 %.4f%%' % ((ratios[0] - 1) * 100))

✅ 改风量 40→100 偏差恒为 -1.6165%


## 6. 结论

1. **两个口径只差一个温度因子**（`T标准/T入口`），代码里是同一份公式的两条前导项，不是两套算法，因此不会各自漂移。
2. **本期平台生效 `eta_excel`**（与 Excel 原表逐字一致），`eta_corrected` 同行输出，待验收后再切换——切换时不需要改代码，只改配置。
3. **数据不可用时宁可输出空值**：质量标记会告诉下游「这一分钟没有可用的效率」，而不是给一个看似合理的数（例如停机时 V_IF 爆表那一段）。
4. 偏差在 20 ℃ 附近为 0、冬季高估、夏季低估，**靠历史对账发现不了**——这也是本期坚持双口径并行输出的原因。